In [2]:
# Cell 1: imports and file paths
import os
import re
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Paths
DATA_LABELED = "data/labeled"
RAW_CSV = os.path.join(DATA_LABELED, "Tweets.csv")   # change filename if you have a full dataset
ARTIFACTS_DIR = "artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print("Files in data/labeled:", os.listdir(DATA_LABELED))
print("Artifacts directory:", ARTIFACTS_DIR)


Files in data/labeled: ['Tweets.csv']
Artifacts directory: artifacts


In [4]:
print(RAW_CSV)

data/labeled\Tweets.csv


In [6]:
# Cell 2: load and inspect
df = pd.read_csv(RAW_CSV)
print("Shape:", df.shape)
display(df.head(10))
print("\nClass distribution:")
print(df['airline_sentiment'].value_counts())

Shape: (14640, 15)


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)
5,570300767074181121,negative,1.0000,Can't Tell,0.6842,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica seriously would pay $30 a fligh...,NaN,2015-02-24 11:14:33 -0800,NaN,Pacific Time (US & Canada)
6,570300616901320704,positive,0.6745,NaN,0.0000,Virgin America,NaN,cjmcginnis,NaN,0,"@VirginAmerica yes, nearly every time I fly VX...",NaN,2015-02-24 11:13:57 -0800,San Francisco CA,Pacific Time (US & Canada)
7,570300248553349120,neutral,0.6340,NaN,NaN,Virgin America,NaN,pilot,NaN,0,@VirginAmerica Really missed a prime opportuni...,NaN,2015-02-24 11:12:29 -0800,Los Angeles,Pacific Time (US & Canada)
8,570299953286942721,positive,0.6559,NaN,NaN,Virgin America,NaN,dhepburn,NaN,0,"@virginamerica Well, I didn't…but NOW I DO! :-D",NaN,2015-02-24 11:11:19 -0800,San Diego,Pacific Time (US & Canada)
9,570295459631263746,positive,1.0000,NaN,NaN,Virgin America,NaN,YupitsTate,NaN,0,"@VirginAmerica it was amazing, and arrived an ...",NaN,2015-02-24 10:53:27 -0800,Los Angeles,Eastern Time (US & Canada)



Class distribution:
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


In [8]:
# Cell 3: preprocessing function (use this same function at inference)
import html
from typing import List

def clean_text(text: str) -> str:
    """
    Lightweight tweet cleaning. Keep this deterministic and identical at train & inference.
    - Lowercase
    - Unescape HTML entities
    - Remove URLs, mentions, RT token
    - Remove extra whitespace and leading/trailing punctuation
    - Keep emoji characters (optional) — here we remove non-alphanum except spaces
    """
    if not isinstance(text, str):
        return ""
    # unescape HTML entities
    text = html.unescape(text)
    text = text.lower()
    # remove URLs
    text = re.sub(r'http\S+|https\S+|www\.\S+', ' ', text)
    # remove mentions and RT
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\brt\b', ' ', text)
    # remove hashtag symbol only (keep the word)
    text = re.sub(r'#', '', text)
    # remove punctuation (keep spaces and alphanumerics)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply to dataframe
df['clean_text'] = df['text'].astype(str).apply(clean_text)
display(df[['text','clean_text','airline_sentiment']].head(10))

,text,clean_text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,what said,neutral
1,@VirginAmerica plus you've added commercials t...,plus you ve added commercials to the experienc...,positive
2,@VirginAmerica I didn't today... Must mean I n...,i didn t today must mean i need to take anothe...,neutral
3,@VirginAmerica it's really aggressive to blast...,it s really aggressive to blast obnoxious ente...,negative
4,@VirginAmerica and it's a really big bad thing...,and it s a really big bad thing about it,negative
5,@VirginAmerica seriously would pay $30 a fligh...,seriously would pay 30 a flight for seats that...,negative
6,"@VirginAmerica yes, nearly every time I fly VX...",yes nearly every time i fly vx this ear worm w...,positive
7,@VirginAmerica Really missed a prime opportuni...,really missed a prime opportunity for men with...,neutral
8,"@virginamerica Well, I didn't…but NOW I DO! :-D",well i didn t but now i do d,positive
9,"@VirginAmerica it was amazing, and arrived an ...",it was amazing and arrived an hour early you r...,positive


In [9]:
# Cell 4: quick stats
df['char_len'] = df['clean_text'].apply(len)
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))
print("Text length stats (chars):")
print(df['char_len'].describe().to_frame().T)
print("\nWord count stats:")
print(df['word_count'].describe().to_frame().T)
print("\nClass distribution (percent):")
print((df['airline_sentiment'].value_counts(normalize=True) * 100).round(2))

Text length stats (chars):
            count       mean       std  min   25%   50%    75%    max
char_len  14640.0  85.984973  35.84457  4.0  58.0  95.0  118.0  165.0

Word count stats:
              count      mean       std  min   25%   50%   75%   max
word_count  14640.0  16.83224  7.155738  1.0  11.0  18.0  23.0  33.0

Class distribution (percent):
airline_sentiment
negative    62.69
neutral     21.17
positive    16.14
Name: proportion, dtype: float64


In [10]:
# Cell 5: TF-IDF fit
# tune max_features if dataset larger; 20000 is reasonable for presentations
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=20000, min_df=3, stop_words='english')

# Fit on cleaned text
X_tfidf = tfidf.fit_transform(df['clean_text'].fillna(''))
print("TF-IDF shape:", X_tfidf.shape)

# Save vectorizer
VECT_PATH = os.path.join(ARTIFACTS_DIR, "tfidf_vectorizer.joblib")
joblib.dump(tfidf, VECT_PATH)
print("Saved TF-IDF vectorizer to:", VECT_PATH)

TF-IDF shape: (14640, 8055)
Saved TF-IDF vectorizer to: artifacts\tfidf_vectorizer.joblib


In [12]:
# Cell 6: train/test split
# map labels to consistent values if needed
label_map = {lab: lab for lab in df['airline_sentiment'].unique()}  # identity map if already good
# if labels are like 0/1/2, ensure dtype
df['airline_sentiment'] = df['airline_sentiment'].astype(str)

X = df['clean_text'].fillna('')
y = df['airline_sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

# Transform into TF-IDF for train/test using the saved vectorizer (ensures same pipeline)
tfidf = joblib.load(VECT_PATH)
X_train_tfidf = tfidf.transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Save processed datasets (sparse matrices can be saved with joblib)
joblib.dump((X_train_tfidf, y_train.reset_index(drop=True)), os.path.join(ARTIFACTS_DIR,'train_tfidf.joblib'))
joblib.dump((X_test_tfidf, y_test.reset_index(drop=True)), os.path.join(ARTIFACTS_DIR,'test_tfidf.joblib'))
print("Saved train/test TF-IDF artifacts to", ARTIFACTS_DIR)

Train: (11712,) Test: (2928,)
Saved train/test TF-IDF artifacts to artifacts


In [13]:
# Cell 7: debug preview - show top TF-IDF features for one sample
def show_top_features(text, tfidf, topn=10):
    vec = tfidf.transform([clean_text(text)])
    arr = vec.toarray()[0]
    top_idx = np.argsort(arr)[-topn:][::-1]
    feat_names = np.array(tfidf.get_feature_names_out())
    for idx in top_idx:
        if arr[idx] > 0:
            print(f"{feat_names[idx]}: {arr[idx]:.4f}")

sample = df['text'].iloc[0]
print("ORIG:", sample)
print("CLEAN:", df['clean_text'].iloc[0])
print("\nTop TF-IDF features for sample:")
show_top_features(sample, tfidf, topn=10)

ORIG: @VirginAmerica What @dhepburn said.
CLEAN: what said

Top TF-IDF features for sample:
said: 1.0000


In [14]:
import os, joblib
import pandas as pd

PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Load the processed DF from earlier steps
# You must reload original CSV & apply clean_text()

# Step 1: load original dataset
df_raw = pd.read_csv("data/labeled/Tweets.csv")

# Step 2: apply clean_text() again if clean_text is defined in your notebook
df_raw['clean_text'] = df_raw['text'].astype(str).apply(clean_text)

# Step 3: basic stats
df_raw['word_count'] = df_raw['clean_text'].apply(lambda x: len(x.split()))

# Step 4: save processed file
df_raw[['text','clean_text','airline_sentiment','word_count']].to_csv(
    os.path.join(PROCESSED_DIR, 'processed_tweets.csv'),
    index=False
)

print("File saved at: data/processed/processed_tweets.csv")


File saved at: data/processed/processed_tweets.csv


In [19]:
# Cell F: sample tweets table with predictions and confidences (saves as PNG)
import pandas as pd
import matplotlib.pyplot as plt

# load processed texts
proc_csv = "data/processed/processed_tweets.csv"
df_proc = pd.read_csv(proc_csv)

# select test indices and get corresponding rows
# we saved y_test as pandas Series earlier; reconstruct index mapping:
_, y_test_ser = joblib.load(os.path.join(ARTIFACTS_DIR, "test_tfidf.joblib"))
# Find sample rows from df_proc that are in test set by matching clean_text & label - approximate
test_texts = list(y_test_ser.index)  # if index alignment preserved
# Simpler: take 8 random test rows
np.random.seed(42)
sample_idx = np.random.choice(df_proc.index, size=8, replace=False)
sample_df = df_proc.loc[sample_idx, ['text','clean_text','airline_sentiment']].reset_index(drop=True)

# For each sample, compute TF-IDF -> meta features -> predict
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = joblib.load(os.path.join(ARTIFACTS_DIR, "tfidf_vectorizer.joblib"))

def predict_stack_for_text(text):
    ct = clean_text(text)
    v = tfidf.transform([ct])
    # base probs
    col = 0
    probs = []
    for name, m in models:
        p = m.predict_proba(v)
        m_classes = list(m.classes_)
        if m_classes != classes:
            reorder = [m_classes.index(c) for c in classes]
            p = p[:, reorder]
        probs.append(p[0])
    probs_concat = np.concatenate(probs)
    pred = meta.predict(probs_concat.reshape(1,-1))[0]
    conf = meta.predict_proba(probs_concat.reshape(1,-1)).max()
    # also record base contributions (max prob per base)
    base_contribs = [round(float(p.max()),3) for p in probs]
    return pred, float(conf), base_contribs

rows = []
for _, r in sample_df.iterrows():
    pred, conf, base_contribs = predict_stack_for_text(r['text'])
    rows.append({
        'text': r['text'],
        'true_label': r['airline_sentiment'],
        'pred_label': pred,
        'pred_conf': round(conf,3),
        'base_contribs': base_contribs
    })
table_df = pd.DataFrame(rows)

# display and save as image
print(table_df)
# save table to png using matplotlib
fig, ax = plt.subplots(figsize=(12, len(table_df)*0.7 + 1))
ax.axis('off')
tbl = ax.table(cellText=table_df.values, colLabels=table_df.columns, cellLoc='left', loc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.2)
plt.title("Sample Tweets — Predictions (Stacking)", fontsize=14)
plt.tight_layout()
plt.savefig("sample_tweets_predictions.png", dpi=200)
plt.show()
print("Saved table image: sample_tweets_predictions.png")


NameError: name 'models' is not defined